# Temporal Congestion Prediction Model

This notebook evaluates a temporal extension of the congestion prediction model.

The baseline model used operational signals from the same month. However, the temporal exploratory analysis suggested that congestion signals may propagate across operational cycles.

To capture this delayed congestion mechanism, lagged versions of key operational indicators are used as predictors.

The model is evaluated using logistic regression with Leave-One-Out Cross Validation (LOOCV) due to the limited dataset size.

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Loading Temporal Feature Dataset

In [2]:
df = pd.read_csv("../../data/JNPA_feature_engineered_temporal.csv")

df.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,...,Total_TEUs_Handled,Date,Rail_Friction,Is_Monsoon,Stress,Risk,Volume_Lag1,Parking_Dwell_Lag1,Rail_Friction_Lag1,Stress_Lag1
0,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,...,467614,2023-02-01,2.900000,0,0,1,522592.0,1.88,4.741758,0.0
1,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,...,560076,2023-03-01,2.856502,0,0,0,467614.0,2.55,2.900000,0.0
2,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,...,503668,2023-04-01,2.094488,0,0,1,560076.0,3.65,2.856502,0.0
3,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,...,538247,2023-05-01,2.965116,0,0,0,503668.0,4.00,2.094488,0.0
4,Jun 23,15.8,13.8,38.4,2.18,2.35,107.0,71.9,69.8,88.5,...,523948,2023-06-01,2.782609,1,523948,0,538247.0,2.33,2.965116,0.0


## Target Variable

In [3]:
y = df["Risk"]

y.value_counts()

Risk
0    25
1     9
Name: count, dtype: int64

## Predictor Features

The temporal model uses lagged operational indicators representing previous-month congestion signals.

In [4]:
features = ["Volume_Lag1","Parking_Dwell_Lag1","Rail_Friction_Lag1","Stress_Lag1"]

X = df[features]

print("Features used:", features)

Features used: ['Volume_Lag1', 'Parking_Dwell_Lag1', 'Rail_Friction_Lag1', 'Stress_Lag1']


## Model Pipeline

Feature scaling is applied using StandardScaler to normalize the predictor variables before training the logistic regression model.

Scaling is implemented inside a pipeline to prevent information leakage during cross-validation.

Class weights are balanced to account for the small number of congestion events in the dataset.

In [5]:
pipeline = Pipeline([("scaler", StandardScaler()),("model", LogisticRegression(class_weight="balanced"))])

## Leave-One-Out Cross Validation

Because the dataset contains a limited number of observations, Leave-One-Out Cross Validation (LOOCV) is used.

Each iteration trains the model on all observations except one and evaluates performance on the remaining observation.

In [6]:
loo = LeaveOneOut()

predictions = []
actual = []

for train_index, test_index in loo.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)

    predictions.append(pred[0])
    actual.append(y_test.values[0])

## Model Performance

In [7]:
print("Accuracy:", accuracy_score(actual, predictions))

print("\nConfusion Matrix")
print(confusion_matrix(actual, predictions))

print("\nClassification Report")
print(classification_report(actual, predictions))

Accuracy: 0.7352941176470589

Confusion Matrix
[[19  6]
 [ 3  6]]

Classification Report
              precision    recall  f1-score   support

           0       0.86      0.76      0.81        25
           1       0.50      0.67      0.57         9

    accuracy                           0.74        34
   macro avg       0.68      0.71      0.69        34
weighted avg       0.77      0.74      0.75        34



## Interpretation of Temporal Model Results

The temporal model was evaluated to determine whether lagged operational signals improve congestion prediction performance.

The model achieved an accuracy of approximately 0.74 with a congestion recall of about 0.67. The confusion matrix shows that the model correctly identifies most normal operational months while successfully detecting a majority of congestion events.

However, the model still produces some false alarms and misses a small number of congestion months. This behaviour is expected given the limited dataset size and the complexity of port logistics operations.

Overall, the model demonstrates a reasonable ability to identify periods of elevated congestion risk using operational indicators derived from the port system.

## Temporal Model Comparison with Baseline Model

The temporal extension model was developed to test whether lagged operational signals improve congestion prediction performance.

However, the results of the temporal model are identical to those obtained from the baseline model. Accuracy, recall, and the confusion matrix structure remain unchanged between the two approaches.

This indicates that the primary congestion signals in the dataset are already captured by the same-month operational indicators used in the baseline model.

While congestion effects may propagate across operational cycles in real port systems, the lagged variables do not provide additional predictive value within the available monthly dataset.

Therefore, the baseline model is retained as the primary predictive framework for the subsequent cause analysis and recommendation layers of the system.